# 1. Importar Librerías

In [2]:
! pip install --user spotipy 

In [3]:
! pip install --user plotly

In [4]:
! pip install --user seaborn

In [5]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv

import requests
import base64

# Librería 1: Matplotlib — visualizaciones básicas y control total
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Librería 2: Seaborn — visualizaciones estadísticas
import seaborn as sns

# Librería 3: Plotly — gráficos interactivos
import plotly.express as px
import plotly.graph_objects as go

# Librería para la API de Spotify 
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

load_dotenv()

c:\Users\Usuario\AppData\Local\Programs\Python\Python312\Lib\site-packages\seaborn\_statistics.py:32: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.4.2)
  from scipy.stats import gaussian_kde


True

# 2. Carga de datos

Para el presente proyecto se utilizarán dos recursos con el objetivo de responder a las preguntas planteadas mediante distintos métodos gráficos. Un primer dataset extraido del siguiente link "https://huggingface.co/datasets/maharshipandya/spotify-tracks-dataset/resolve/main/dataset.csv", y un segundo conjunto de datos obtenido gracias a la API Key de spotify developer. 

### 2.1 Carga de archivos csv

A parte de cargar los datos se hará un pequeño estudio del formato que presentan.

In [6]:
URL_DATASET = 'https://huggingface.co/datasets/maharshipandya/spotify-tracks-dataset/resolve/main/dataset.csv'

dataset_csv = pd.read_csv(URL_DATASET)
dataset_csv.head(5)

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [7]:
dataset_csv.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        114000 non-null  int64  
 1   track_id          114000 non-null  object 
 2   artists           113999 non-null  object 
 3   album_name        113999 non-null  object 
 4   track_name        113999 non-null  object 
 5   popularity        114000 non-null  int64  
 6   duration_ms       114000 non-null  int64  
 7   explicit          114000 non-null  bool   
 8   danceability      114000 non-null  float64
 9   energy            114000 non-null  float64
 10  key               114000 non-null  int64  
 11  loudness          114000 non-null  float64
 12  mode              114000 non-null  int64  
 13  speechiness       114000 non-null  float64
 14  acousticness      114000 non-null  float64
 15  instrumentalness  114000 non-null  float64
 16  liveness          11

In [8]:
dataset_csv.isnull().sum()[dataset_csv.isnull().sum() > 0]

artists       1
album_name    1
track_name    1
dtype: int64

Al presentar valores vacios en el apartado de limpieza del dataset se procederá a eliminarlos

### 2.2 Spotify Web API

Para poder usar la API de spotify debemos obtener el token para interactuar con esta.

In [ ]:
CLIENT_ID = os.getenv('CLIENT_ID')
CLIENT_SECRET = os.getenv('CLIENT_SECRET')

def get_token():
    response = requests.post(
        url = 'https://accounts.spotify.com/api/token',
        headers = {
            'content-type' : 'application/x-www-form-urlencoded',
            'Authorization' : f'Basic {base64.b64encode(f'{CLIENT_ID}:{CLIENT_SECRET}'.encode()).decode()}'
        },
        data={'grant_type': 'client_credentials'}
    )

    response.raise_for_status()
    return response.json()['access_token']

token = get_token()

'BQAvj1sMfeBNZ5Nwoav4yF_-v3w6bWBPVQLJVTMbX0cX47teFS89CkSUhU0PcPnvmhrJo4QGLNZptgu9i4H1iYd9Q3_p6PCQdvGklf7UHEv7YayybhJfsMpuXK_xC-u8XVEPXuY3vbo'

Obtenido el token, vamos hacer una prueba con las Top 50 canciones globales